In [47]:
%load_ext autoreload
%autoreload 2

from io import BytesIO
from pathlib import Path

from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from PIL import Image
import numpy as np
from typing import List
import time
import torch

from AnytimeTrajectoryPredictor.models.ObjectTracker import ObjectTracker

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
repo_root = Path.cwd().parent
images_parquet = repo_root / 'data/waymo/images.parquet'

if not images_parquet.exists():
    raise FileNotFoundError(f'File not found: {images_parquet}')

df = pd.read_parquet(images_parquet)
print(f'Number of rows: {len(df)}')
print('Columns:', list(df.columns))

required = {'image_jpeg'}
missing = required - set(df.columns)
if missing:
    raise KeyError(f'Missing required columns: {missing}')

Number of rows: 2810
Columns: ['split', 'image_id', 'scene_id', 'frame_timestamp_micros', 'camera_name', 'camera_name_text', 'image_jpeg', 'image_format', 'image_width', 'image_height', 'visible_trajectory_ids', 'num_visible_trajectories']


In [48]:
def display_results(
    image: Image.Image, 
    features: np.ndarray,  # (1, O, F) tensor
    mask: np.ndarray,      # (1, O, 1) tensor
    feature_components: List[str],
    inference_time: float,
):
    """
    Plot bounding boxes from ObjectTracker features.
    
    Assumes bboxes are always the first component (4 dims: center_x, center_y, width, height).
    Extracts confidences and class_ids if present.
    """
    if feature_components is None:
        feature_components = ["bboxes"]
    
    # Squeeze batch dimension: (1, O, F) -> (O, F)
    features_squeezed = features.squeeze(0)  # (O, F)
    mask_squeezed = mask.squeeze(0)  # (O, 1)
    
    # Count valid objects
    valid_objects = int(mask_squeezed.sum().item())
    if valid_objects == 0:
        print("No valid detections to display.")
        return
    
    # Filter by mask
    valid_idx = mask_squeezed[:, 0] > 0.5
    features_valid = features_squeezed[valid_idx]  # (valid_count, F)
    
    # Extract bboxes (always first 4 columns, in center/size format)
    bbox_data = features_valid[:, :4]  # (valid_count, 4)
    center_x, center_y, width, height = bbox_data[:, 0], bbox_data[:, 1], bbox_data[:, 2], bbox_data[:, 3]
    
    # Convert center/size to xyxy
    x1 = center_x - width / 2
    y1 = center_y - height / 2
    x2 = center_x + width / 2
    y2 = center_y + height / 2
    boxes_xyxy = np.stack([x1, y1, x2, y2], axis=1)  # (valid_count, 4)
    
    # Extract confidences if present
    confidences = None
    conf_idx = None
    if "confidences" in feature_components:
        conf_idx = 4 + feature_components.index("confidences")
        confidences = features_valid[:, conf_idx-1]
    
    # Extract class_ids if present
    class_ids = None
    cls_idx = None
    if "class_ids" in feature_components:
        cls_idx = 4 + feature_components.index("class_ids")
        class_ids = features_valid[:, cls_idx-1]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 7), gridspec_kw={'width_ratios': [3, 1]})
    
    ax = axes[0]
    emb_ax = axes[1]

    ax.imshow(image)
    ax.axis('off')
    
    for i, box in enumerate(boxes_xyxy):
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        
        rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        
        # Build label
        class_name = "na"
        if class_ids is not None and i < len(class_ids):
            class_name = str(class_ids[i])
        
        score_txt = f"{float(confidences[i]):.2f}" if (confidences is not None and i < len(confidences)) else "na"
        label = f"{class_name} | conf={score_txt}"
        
        ax.text(
            x1,
            max(0, y1 - 6),
            label,
            fontsize=9,
            color='black',
            bbox=dict(facecolor='lime', alpha=0.75, edgecolor='none', pad=1.5),
        )

    if 'embeddings' in feature_components:
        # Extract and print embeddings from features_valid
        embed_idx = 4 + feature_components.index("embeddings")
        embeddings = features_valid[:, embed_idx-1:]  # Assuming embeddings are the last components
        # Check that all embeddings for objects are the same (since they are scene-level features)
        all_same = np.all(np.isclose(embeddings - embeddings[0], 0, atol=1e-6))
        if not all_same:
            print("Warning: Embeddings vary across objects, which is unexpected for scene-level features.")

    img_shape = image.size if hasattr(image, 'size') else image.shape
    ax.set_title(f'{valid_objects} detections - img ({', '.join(map(str, img_shape))}) - embed dim {embeddings.shape[1]} - inference {inference_time:.2f}ms')

    # Show embeddings as a heatmap (take first object's embeddings, make it 2D for visualization)
    if 'embeddings' in feature_components:
        embed_idx = 4 + feature_components.index("embeddings")
        embeddings = features_valid[:, embed_idx-1:]  # (valid_count, embedding_dim)
        if embeddings.shape[0] > 0: # reshape by taking the sqrt of the embedding dimension (NOT assuming it's a perfect square, round to nearest int)
            embedding_dim = embeddings.shape[1]
            embed_size = int(np.ceil(np.sqrt(embedding_dim)))
            embedding_2d = np.zeros((embed_size, embed_size))
            embedding_2d.flat[:embedding_dim] = embeddings[0]  # Take the first object's embeddings for visualization
            im = emb_ax.imshow(embedding_2d, cmap='viridis')
            emb_ax.set_title('YOLO Embeddings')

    return fig, ax, emb_ax


In [49]:
# Models
features_used = ["bboxes", "confidences", "class_ids", "embeddings"]

model = ObjectTracker(
    feature_components=features_used,
    imgsz=1920
)

In [50]:
# Create sequences from all cam 1s
cam1s = df[df['camera_name'] == 1]
cam1s = cam1s.loc[:, ['scene_id', 'frame_timestamp_micros', 'image_jpeg']]

scenes = {}

cam1s.sort_values(by=['scene_id', 'frame_timestamp_micros'], inplace=True)
for scene_id in tqdm(cam1s['scene_id'].unique()):
    group = cam1s[cam1s['scene_id'] == scene_id]
    group.sort_values(by='frame_timestamp_micros', inplace=True)
    scenes[scene_id] = group['image_jpeg'].tolist()

print(f"Total scenes: {len(scenes)}")

100%|██████████| 3/3 [00:00<00:00, 1454.17it/s]

Total scenes: 3


In [ ]:

np.random.seed(42)  # For reproducibility
rand_ids = np.random.choice(df.index, size=5, replace=False)

# Create a GIF of outputs for each scene
for scene in scenes.keys():
    buffer = []

    model = ObjectTracker(
        feature_components=features_used,
        imgsz=1920,
        verbose=False
    )

    N_FRAMES = 25

    # Pick N random consecutive frames from the scene
    original_len = len(scenes[scene])
    scene_frames = scenes[scene]
    if len(scene_frames) > N_FRAMES:
        start_idx = np.random.randint(0, len(scene_frames) - N_FRAMES)
        scene_frames = scene_frames[start_idx:start_idx + N_FRAMES]

    print(f"Scene {scene}: select {start_idx} to {start_idx + N_FRAMES} among {original_len} frames")

    for i, img_jpeg in enumerate(scene_frames):
        print(f"Processing scene {scene}, frame {i+1}/{len(scene_frames)}", end='\r' if i < len(scene_frames) - 1 else '\n')

        image = Image.open(BytesIO(img_jpeg)).convert("RGB")

        start_time = time.time()
        output = model(image)
        inf_time = time.time() - start_time
        inf_time_ms = inf_time * 1000

        features = output["features"].cpu().numpy()  # (1, O, F)
        mask = output["mask"].cpu().numpy()  # (1, O, 1)
        
        valid_count = int(mask.sum())
        
        out = display_results(
            image, 
            features, 
            mask, 
            feature_components=list(model.feature_components),
            inference_time=inf_time_ms
        )

        if out is not None:
            fig, ax, emb_ax = out
            buf = BytesIO()
            plt.savefig(buf, format='png')
            buf.seek(0)
            buffer.append(Image.open(buf))
            plt.close(fig)

    if buffer:
        gif_folder = repo_root / 'outputs'
        gif_folder.mkdir(exist_ok=True)

        gif_path = gif_folder / f'scene_{scene}_output.gif'
        buffer[0].save(gif_path, save_all=True, append_images=buffer[1:], duration=N_FRAMES*10, loop=0)
        print(f"Saved GIF for scene {scene} at {gif_path}")


Scene 10017090168044687777_6380_000_6400_000: select 119 to 144 among 198 frames
Processing scene 10017090168044687777_6380_000_6400_000, frame 25/25
Saved GIF for scene 10017090168044687777_6380_000_6400_000 at /Users/nathangromb/Documents/MA4/VI/project/outputs/scene_10017090168044687777_6380_000_6400_000_output.gif
Scene 10203656353524179475_7625_000_7645_000: select 78 to 103 among 198 frames
Processing scene 10203656353524179475_7625_000_7645_000, frame 25/25
Saved GIF for scene 10203656353524179475_7625_000_7645_000 at /Users/nathangromb/Documents/MA4/VI/project/outputs/scene_10203656353524179475_7625_000_7645_000_output.gif
Scene 10504764403039842352_460_000_480_000: select 115 to 140 among 166 frames
